# 6 WorkFlow Gerencial, futuro=SEP

### 6.1 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán enriquecer

#### 6.2  Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab
*   Bajar el **dataset_historico** al Google Drive y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/utn2026-b40a/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"
descargar  "gerencial_competencia_2026.csv.gz"


## 6.3  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [188]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sun Sep 20 10:27:00 PM 2026"

In [189]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1930152,103.1,4836504,258.3,4836504,258.3
Vcells,3920508,30.0,381608751,2911.5,477807357,3645.4


In [190]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

#### Parametros
Si es gerente, no cambie nada
<br>Si es Analista, cambie el nombre del dataset

In [191]:
PARAM <- list()
PARAM$semilla_primigenia <- 679957

PARAM$experimento <- 6306
PARAM$dataset <- "gerencial_competencia_2026.csv.gz"

#### Carpeta del Experimento

In [192]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

### 6.3.1   Preprocesamiento del dataset

#### 6.3.1.1  DT incorporar dataset

In [193]:
# lectura del dataset
dataset <- fread(paste0("/content/datasets/", PARAM$dataset))

#### 6.3.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [194]:
dataset[ foto_mes==202006, internet:=NA]
dataset[ foto_mes==202006, mrentabilidad:=NA]
dataset[ foto_mes==202006, mrentabilidad_annual:=NA]
dataset[ foto_mes==202006, mcomisiones:=NA]
dataset[ foto_mes==202006, mactivos_margen:=NA]
dataset[ foto_mes==202006, mpasivos_margen:=NA]
dataset[ foto_mes==202006, mcuentas_saldo:=NA]
dataset[ foto_mes==202006, ctarjeta_visa_transacciones:=NA]
dataset[ foto_mes==202006, mtarjeta_visa_consumo:=NA]
dataset[ foto_mes==202006, mtarjeta_master_consumo:=NA]
dataset[ foto_mes==202006, ccallcenter_transacciones:=NA]
dataset[ foto_mes==202006, chomebanking_transacciones:=NA]
dataset[ foto_mes==202006, chomebanking_transacciones:=NA]

#### 6.3.1.3  DR  Data Drifting
Se intenta corregir el data drifting, quizas ajustando por IPC ...
<br>Esta parte podrá ser abordada por todos los Analistas y también la Gerenciapero se decide pedagogicamente no incluirla en esta primer version para reducir la carga cognitiva

In [195]:
drift_rank_cero_fijo <- function(campos_drift) {
  cat("Inicio drift_rank_cero_fijo()\n")

  for (campo in campos_drift) {
    cat(campo, " ")

    dataset[, paste0(campo, "_rank") := 0.0]

    dataset[get(campo) > 0,
            paste0(campo, "_rank") := frank(get(campo), ties.method = "random") / .N,
            by = foto_mes]

    dataset[get(campo) < 0,
            paste0(campo, "_rank") := -frank(-get(campo), ties.method = "random") / .N,
            by = foto_mes]

    dataset[is.na(get(campo)), paste0(campo, "_rank") := NA_real_]

    dataset[, (campo) := get(paste0(campo, "_rank"))]
    dataset[, paste0(campo, "_rank") := NULL]
  }

  cat("\nFin drift_rank_cero_fijo()\n")
}

campos_monetarios <- colnames(dataset)
campos_monetarios <- campos_monetarios[campos_monetarios %like% "^(m|Visa_m|Master_m|vm_m)"]
campos_monetarios <- setdiff(campos_monetarios, c("Master_fechaalta", "Visa_fechaalta"))

setorder(dataset, numero_de_cliente, foto_mes)

drift_rank_cero_fijo(campos_monetarios)

Inicio drift_rank_cero_fijo()
mrentabilidad  mrentabilidad_annual  mcomisiones  mactivos_margen  mpasivos_margen  mcuenta_corriente  mcaja_ahorro  mcuentas_saldo  mtarjeta_visa_consumo  mtarjeta_master_consumo  mprestamos_personales  mpayroll  Master_mpagominimo  Visa_mpagominimo  
Fin drift_rank_cero_fijo()


#### 6.3.1.3  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

In [196]:
# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

# ---------------------------------------------------------------------
# division segura
# Evita Inf/NaN cuando el denominador es 0:
# denominador = 0 -> NA
# ---------------------------------------------------------------------
divseg <- function( numerador, denominador )
{
  denominador[ denominador == 0 ] <- NA
  return( numerador / denominador )
}


# ---------------------------------------------------------------------
# VARIABLES BASICAS
# ---------------------------------------------------------------------

# el mes 1,2, ..12
if( atributos_presentes( c("foto_mes") ))
  dataset[, kmes := foto_mes %% 100]

# variable extraida de una tesis de maestria de Irlanda
if( atributos_presentes( c("mpayroll", "cliente_edad") ))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]

# ---------------------------------------------------------------------
# FE intra-mes basado en Feature Importance del baseline (10 semillas)
#
# Se toman las 10 variables mas importantes del baseline, EXCLUYENDO
# ctrx_quarter_lag1 porque no es intra-mes y proviene del FE historico.
# ---------------------------------------------------------------------

vars_top10 <- c(
  "ctrx_quarter",
  "mcaja_ahorro",
  "cpayroll_trx",
  "mcuentas_saldo",
  "mcuenta_corriente",
  "mprestamos_personales",
  "mtarjeta_visa_consumo",
  "Visa_mpagominimo",
  "cliente_edad",
  "mpasivos_margen"
)

# Variables de alto riesgo como denominadores:
# pueden presentar 0 o NA porque no todos los clientes tienen
# necesariamente esos productos.
vars_alto_riesgo <- c(
  "cpayroll_trx",
  "mcuenta_corriente",
  "mprestamos_personales",
  "Visa_mpagominimo"
)


# ---------------------------------------------------------------------
# A) FLAGS EXPLICITOS
# ---------------------------------------------------------------------
if( atributos_presentes( vars_alto_riesgo ) )
{
  for( v in vars_alto_riesgo )
  {
    nombre_flag <- paste0( "flag_tiene_", v )

    dataset[, (nombre_flag) := as.integer( get(v) > 0 ) ]
  }
}


# ---------------------------------------------------------------------
# B) ESTANDARIZACION DE LAS 10 VARIABLES TOP
#
# IMPORTANTE:
# La media y el desvio estandar se calculan SOLO con el periodo de
# entrenamiento (202005-202104), para evitar leakage de validation/future.
# Luego esos mismos parametros se aplican a todos los registros.
#
# Las variables estandarizadas se llaman z_<variable>.
# ---------------------------------------------------------------------

if( atributos_presentes( vars_top10 ) )
{
  filas_estandarizacion <- dataset[foto_mes %in% c(
    202005, 202006, 202007, 202008, 202009, 202010,
    202011, 202012, 202101, 202102, 202103, 202104
  )]

  for( v in vars_top10 )
  {
    media_v <- mean( filas_estandarizacion[[v]], na.rm = TRUE )
    desvio_v <- sd( filas_estandarizacion[[v]], na.rm = TRUE )

    nombre_z <- paste0( "z_", v )

    if( is.finite(desvio_v) && desvio_v > 0 )
    {
      dataset[, (nombre_z) := (get(v) - media_v) / desvio_v ]
    }
    else
    {
      dataset[, (nombre_z) := NA_real_ ]
    }

    cat(sprintf(
      "Estandarizada %s | media entrenamiento = %.6f | sd entrenamiento = %.6f\n",
      v, media_v, desvio_v
    ))
  }

  rm(filas_estandarizacion)
}

# ---------------------------------------------------------------------
# C) 45 RATIOS UNICOS SOBRE VARIABLES ESTANDARIZADAS
#
# Se genera UNA sola division por cada par de variables:
#   z_A / z_B
# y NO se genera z_B / z_A.
#
# Esto produce 10*9/2 = 45 ratios unicos.
#
# ATENCION: como los z pueden estar cerca de 0, estos ratios pueden
# tomar valores extremos. divseg() convierte denominadores exactamente
# iguales a 0 en NA. LightGBM puede manejar NA nativamente.
# ---------------------------------------------------------------------

vars_z_top10 <- paste0( "z_", vars_top10 )

if( atributos_presentes( vars_z_top10 ) )
{
  n_vars <- length( vars_z_top10 )

  for( i in 1:(n_vars - 1) )
  {
    for( j in (i + 1):n_vars )
    {
      num <- vars_z_top10[i]
      den <- vars_z_top10[j]

      nombre_nueva <- paste0(
        "ratio_",
        vars_top10[i],
        "_sobre_",
        vars_top10[j],
        "_z"
      )

      dataset[, (nombre_nueva) := divseg(
        get(num),
        get(den)
      )]
    }
  }
}



Estandarizada ctrx_quarter | media entrenamiento = 107.457588 | sd entrenamiento = 81.531777
Estandarizada mcaja_ahorro | media entrenamiento = 0.466123 | sd entrenamiento = 0.305764
Estandarizada cpayroll_trx | media entrenamiento = 0.946460 | sd entrenamiento = 1.369012
Estandarizada mcuentas_saldo | media entrenamiento = 0.314322 | sd entrenamiento = 0.481866
Estandarizada mcuenta_corriente | media entrenamiento = -0.164499 | sd entrenamiento = 0.366493
Estandarizada mprestamos_personales | media entrenamiento = 0.114023 | sd entrenamiento = 0.251053
Estandarizada mtarjeta_visa_consumo | media entrenamiento = 0.425004 | sd entrenamiento = 0.320502
Estandarizada Visa_mpagominimo | media entrenamiento = 0.415959 | sd entrenamiento = 0.322953
Estandarizada cliente_edad | media entrenamiento = 46.728670 | sd entrenamiento = 12.971815
Estandarizada mpasivos_margen | media entrenamiento = 0.450576 | sd entrenamiento = 0.354924


In [197]:
# visualizo las columas del dataset a esta etapa
colnames(dataset)

[1] "numero_de_cliente"                                        
 [2] "foto_mes"                                                 
 [3] "internet"                                                 
 [4] "cliente_edad"                                             
 [5] "cliente_antiguedad"                                       
 [6] "mrentabilidad"                                            
 [7] "mrentabilidad_annual"                                     
 [8] "mcomisiones"                                              
 [9] "mactivos_margen"                                          
[10] "mpasivos_margen"                                          
[11] "cproductos"                                               
[12] "mcuenta_corriente"                                        
[13] "mcaja_ahorro"                                             
[14] "cdescubierto_preacordado"                                 
[15] "mcuentas_saldo"                                           
[16] "ctarjeta_visa_transacciones"                              
[17] "mtarjeta_visa_consumo"                                    
[18] "mtarjeta_master_consumo"                                  
[19] "mprestamos_personales"                                    
[20] "cpayroll_trx"                                             
[21] "mpayroll"                                                 
[22] "ccomisiones_mantenimiento"                                
[23] "ccallcenter_transacciones"                                
[24] "chomebanking_transacciones"                               
[25] "ctrx_quarter"                                             
[26] "Master_status"                                            
[27] "Master_fechaalta"                                         
[28] "Master_mpagominimo"                                       
[29] "Visa_status"                                              
[30] "Visa_fechaalta"                                           
[31] "Visa_mpagominimo"                                         
[32] "clase_ternaria"                                           
[33] "kmes"                                                     
[34] "mpayroll_sobre_edad"                                      
[35] "flag_tiene_cpayroll_trx"                                  
[36] "flag_tiene_mcuenta_corriente"                             
[37] "flag_tiene_mprestamos_personales"                         
[38] "flag_tiene_Visa_mpagominimo"                              
[39] "z_ctrx_quarter"                                           
[40] "z_mcaja_ahorro"                                           
[41] "z_cpayroll_trx"                                           
[42] "z_mcuentas_saldo"                                         
[43] "z_mcuenta_corriente"                                      
[44] "z_mprestamos_personales"                                  
[45] "z_mtarjeta_visa_consumo"                                  
[46] "z_Visa_mpagominimo"                                       
[47] "z_cliente_edad"                                           
[48] "z_mpasivos_margen"                                        
[49] "ratio_ctrx_quarter_sobre_mcaja_ahorro_z"                  
[50] "ratio_ctrx_quarter_sobre_cpayroll_trx_z"                  
[51] "ratio_ctrx_quarter_sobre_mcuentas_saldo_z"                
[52] "ratio_ctrx_quarter_sobre_mcuenta_corriente_z"             
[53] "ratio_ctrx_quarter_sobre_mprestamos_personales_z"         
[54] "ratio_ctrx_quarter_sobre_mtarjeta_visa_consumo_z"         
[55] "ratio_ctrx_quarter_sobre_Visa_mpagominimo_z"              
[56] "ratio_ctrx_quarter_sobre_cliente_edad_z"                  
[57] "ratio_ctrx_quarter_sobre_mpasivos_margen_z"               
[58] "ratio_mcaja_ahorro_sobre_cpayroll_trx_z"                  
[59] "ratio_mcaja_ahorro_sobre_mcuentas_saldo_z"                
[60] "ratio_mcaja_ahorro_sobre_mcuenta_corriente_z"             
[61] "ratio_mcaja_ahorro_sobre_mprestamos_personales_z"         
[62] "ratio_mcaja_ahorro_sobre_mtarj

#### 6.3.1.4  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest

Esto se mostrará unicamente a la *modalidad Analista Sr*

In [198]:
# No se implementa Feature Engineering a partir de Random Forest

#### 6.3.1.5  FEhist Feature Engineering historico

El Fature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

In [199]:
# Feature Engineering Historico

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
) )

# https://rdrr.io/cran/data.table/man/shift.html

# lags de orden 1
dataset[,
    paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# lags de orden 2
dataset[,
    paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# lags de orden 3
dataset[,
    paste0(cols_lagueables, "_lag3") :=
        shift(.SD, 3, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# agrego los delta lags
for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
    dataset[, paste0(vcol, "_delta3") := get(vcol) - get(paste0(vcol, "_lag3"))]
}

# MEDIAS MÓVILES (frollmean) - ventana de 3 meses
# align = "right": toma el mes actual y N-1 atrás
dataset[, paste0(cols_lagueables, "_ma3") := frollmean(.SD, n = 3, align = "right", fill = NA, na.rm = TRUE), by = numero_de_cliente, .SDcols = cols_lagueables]


Verificacion de los campos recien creados

In [200]:
ncol(dataset)
colnames(dataset)

[1] 723

[1] "numero_de_cliente"                                               
  [2] "foto_mes"                                                        
  [3] "internet"                                                        
  [4] "cliente_edad"                                                    
  [5] "cliente_antiguedad"                                              
  [6] "mrentabilidad"                                                   
  [7] "mrentabilidad_annual"                                            
  [8] "mcomisiones"                                                     
  [9] "mactivos_margen"                                                 
 [10] "mpasivos_margen"                                                 
 [11] "cproductos"                                                      
 [12] "mcuenta_corriente"                                               
 [13] "mcaja_ahorro"                                                    
 [14] "cdescubierto_preacordado"                                        
 [15] "mcuentas_saldo"                                                  
 [16] "ctarjeta_visa_transacciones"                                     
 [17] "mtarjeta_visa_consumo"                                           
 [18] "mtarjeta_master_consumo"                                         
 [19] "mprestamos_personales"                                           
 [20] "cpayroll_trx"                                                    
 [21] "mpayroll"                                                        
 [22] "ccomisiones_mantenimiento"                                       
 [23] "ccallcenter_transacciones"                                       
 [24] "chomebanking_transacciones"                                      
 [25] "ctrx_quarter"                                                    
 [26] "Master_status"                                                   
 [27] "Master_fechaalta"                                                
 [28] "Master_mpagominimo"                                              
 [29] "Visa_status"                                                     
 [30] "Visa_fechaalta"                                                  
 [31] "Visa_mpagominimo"                                                
 [32] "clase_ternaria"                                                  
 [33] "kmes"                                                            
 [34] "mpayroll_sobre_edad"                                             
 [35] "flag_tiene_cpayroll_trx"                                         
 [36] "flag_tiene_mcuenta_corriente"                                    
 [37] "flag_tiene_mprestamos_personales"                                
 [38] "flag_tiene_Visa_mpagominimo"                                     
 [39] "z_ctrx_quarter"                                                  
 [40] "z_mcaja_ahorro"                                                  
 [41] "z_cpayroll_trx"                                                  
 [42] "z_mcuentas_saldo"                                                
 [43] "z_mcuenta_corriente"                                             
 [44] "z_mprestamos_personales"                                         
 [45] "z_mtarjeta_visa_consumo"                                         
 [46] "z_Visa_mpagominimo"                                              
 [47] "z_cliente_edad"                                                  
 [48] "z_mpasivos_margen"                                               
 [49] "ratio_ctrx_quarter_sobre_mcaja_ahorro_z"                         
 [50] "ratio_ctrx_quarter_sobre_cpayroll_trx_z"                         
 [51] "ratio_ctrx_quarter_sobre_mcuentas_saldo_z"                       
 [52] "ratio_ctrx_quarter_sobre_mcuenta_corriente_z"                    
 [53] "ratio_ctrx_quarter_sobre_mprestamos_personales_z"                
 [54] "ratio_ctrx_quarter_sobre_mtarjeta_visa_consumo_z"                
 [55] "ratio_ctrx_quarter_sobre_Visa_mpagominimo_z"         

#### 6.3.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr* por algun canal secreto de forma de no confundir a los *Analista Jr*  nni distraer con detalles operativos a la estratégica *Modalidad Gerencial*

In [201]:
# No se implementa la reduccion de la dimensionalidad con canaritos

### 6.3.2 Modelado

#### 6.3.2.1 Training Strategy

Esta etapa de Workflow de  Training Strategy esta pensada para la *Modalidad Gerencial* que posee el dataset reducido de [202005, 202109]
<br> Si usted es un Analista, posee el periodo de [201901, 202109] y deberá experimentar en que meses le conviene experimentar

<br> A la *Modalidad Gerencial* no se le complicada la vida con el undersampling de los continua, por eso PARAM$trainingstrategy$training_pct <- 1.0
<br> Sin embargo, si usted es  *Analista SR* posee un dataset 50 veces ( filas x columnas) más grande que la *Modalidad Gerencial*  y por un tema de velocidad y experimentación más rápida puede llegar a necesitar activar el undersampling de la clase mayoritaria, a pesar de estar corriendo en Google Cloud.

Se hace una estrategia de entrenamiento muy sencilla, tomando todos los meses posibles, SIN eliminar nada x pandemia ni por ningun otro motivo

* future = 202109  obviamente completo

* final_train =  [ 202005, 202107 ]  SIN undersampling

* training
   * testing = NO HAY
   * validation =  202107   completo, sin undersampling
   * training = [ 202005, 202106 ]  donde se consideran el 100% de los CONTINUA

In [202]:
PARAM$trainingstrategy$validate <- c(202107)

PARAM$trainingstrategy$training <- c(
  202106, 202105, 202104, 202103, 202102, 202101,
  202012, 202011, 202010, 202009, 202008, 202007,
  202006, 202005
)

PARAM$trainingstrategy$training_pct <- 1.0


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [203]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

In [204]:
# los campos en los que se entrena
campos_buenos <- copy( setdiff(
    colnames(dataset), c("clase_ternaria","clase01","azar"))
)

Esta celda tarda en correr interminables 7 minutos en Colab
<br> ya que debe instalar la librería de LightGBM

In [205]:
# preparo para que se puede hacer undersampling de los CONTINUA
#  solamente por un tema de VELOCIDAD
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar:=runif(nrow(dataset))]

# undersampling de los CONTINUA
dataset[, fold_train :=  foto_mes %in%  PARAM$trainingstrategy$training &
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
     azar < PARAM$trainingstrategy$training_pct ) ]


if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

dtrain <- lgb.Dataset(
  data= data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
  label= dataset[fold_train == TRUE, clase01],
  free_raw_data= TRUE
)

In [206]:
# datos de validation
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
  label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
  free_raw_data= TRUE
)

nrow(dvalidate)

[1] 13202

####  6.3.2.2. Hyperparameter Tuning

* Clase binaria que se optimiza :  positivos = [ BAJA+1, BAJA+2 ]

* Metrica que se optimiza **AUC** Area Under Curve de la  ROC Curve

es muy importante notar que intencionalmente  **NO** se está optimizando la funcion de ganancia del problema

* Parametros no default, fijos de LightGBM que no se optimizan
  * max_bin = 31 , Alienigenas Ancestrales contruyeron las pirámides y dejaron a la humanidad en un jeroglifico  *max_bin=31*
  * feature_fraction = 0.5  para poner algo que generalmente no falla
  * learning_rate = 0.03  para que aprenda lento


* Parametros que se optimizan en el Grid Search
  * num_leaves  [64, 512]
  * min_data_in_leaf  [64, 2048]

In [207]:
# parametros fijos del LightGBM
PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE, # para evitar warning
  seed= PARAM$semilla_primigenia,
  max_bin= 31,
  learning_rate= 0.03,
  feature_fraction= 0.5,
  num_iterations= 2048,  # valor grande, lo limita early_stopping_rounds
  early_stopping_rounds= 200,
  num_leaves= 64,
  min_data_in_leaf= 128
)


In [208]:
# En  x llegan los parametros moviles de LightGBM
#  devuelve la AUC en validate del modelo entrenado
#  en el parametro x llegan los hiperparámetros que se estan optimizando

Estimar_AUC_lightgbm <- function(x) {

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # entreno LightGBM
  modelo_train <- lgb.train(
    data= dtrain,
    valids= list(valid = dvalidate),
    eval= "auc",
    param= param_completo,
    verbose= -100
  )

  # recupero la AUC en validation
  AUC <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]

  message(format(Sys.time(), "%a %b %d %X %Y  "),
    toString(x),
    " niter ", modelo_train$best_iter,
    " AUC ", AUC
  )

  niter <- modelo_train$best_iter
  # hago espacio en la memoria
  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  return( list(AUC, niter))
}

seteo del Grid Search

In [209]:
# lo que sigue a continuacion es una forma alternativa a los loops anidados
# creo una tabla con el producto cartesiano de los vectores
tb_nueva <- CJ(
  num_leaves= c(64, 128, 256, 512),
  min_data_in_leaf= c(64, 256, 512, 1024, 2048)
)

Corrida del Grid Search,  aqui se hace el trabajo pesado
<br> por favor no se asuste con los warnings que pudieran aparecer
<br> ATENCION, la siguiente celda demora 50 minutos en Colab
<br> lamento profundamente tal intolerable espera gerencial

In [210]:
# registro a registro calculo la AUC
tb_nueva[, c("AUC", "num_iterations"):= Estimar_AUC_lightgbm( .SD ),
  by=1:nrow(tb_nueva) ]

Sun Sep 20 10:30:33 PM 2026  64, 64 niter 178 AUC 0.943961308652418

Sun Sep 20 10:33:06 PM 2026  64, 256 niter 174 AUC 0.948548315386534

Sun Sep 20 10:35:34 PM 2026  64, 512 niter 151 AUC 0.949040198661852

Sun Sep 20 10:38:40 PM 2026  64, 1024 niter 230 AUC 0.947193250702148

Sun Sep 20 10:42:10 PM 2026  64, 2048 niter 250 AUC 0.948621620742379

Sun Sep 20 10:47:22 PM 2026  128, 64 niter 439 AUC 0.949578494203889

Sun Sep 20 10:50:18 PM 2026  128, 256 niter 149 AUC 0.948359196243643

Sun Sep 20 10:53:53 PM 2026  128, 512 niter 218 AUC 0.947766247004348

Sun Sep 20 10:56:56 PM 2026  128, 1024 niter 153 AUC 0.94823557478556

Sun Sep 20 11:00:04 PM 2026  128, 2048 niter 202 AUC 0.948319290369455

Sun Sep 20 11:05:47 PM 2026  256, 64 niter 377 AUC 0.951737315245562

Sun Sep 20 11:12:59 PM 2026  256, 256 niter 515 AUC 0.951530411963087

Sun Sep 20 11:16:46 PM 2026  256, 512 niter 180 AUC 0.95029593242027

Sun Sep 20 11:21:00 PM 2026  256, 1024 niter 272 AUC 0.948722252946854

Sun Sep 20 

la optimizacion de hiperparámetros de tipo  Grid Search ha corrido, extraigo los mejores hiperparametros

In [211]:
tb_nueva

fwrite( tb_nueva,
  file= "tb_grid_search_01.txt",
  sep="\t",
  append= TRUE
)

num_leaves,min_data_in_leaf,AUC,num_iterations
<dbl>,<dbl>,<dbl>,<int>
64,64,0.9439613,178
64,256,0.9485483,174
64,512,0.9490402,151
64,1024,0.9471933,230
64,2048,0.9486216,250
128,64,0.9495785,439
128,256,0.9483592,149
128,512,0.9477662,218
128,1024,0.9482356,153


In [212]:
setorder( tb_nueva, -AUC)  # ordeno DESCENDENTE por AUC
PARAM$out$lgbm$AUC <- tb_nueva[1, AUC] # en la posicion 1 estan los mejores
PARAM$out$lgbm$mejores_hiperparametros <- as.list( tb_nueva[1] )
PARAM$out$lgbm$mejores_hiperparametros$AUC <- NULL
PARAM$out$lgbm$mejores_hiperparametros

$num_leaves
[1] 512

$min_data_in_leaf
[1] 64

$num_iterations
[1] 526

### 6.3.3 Produccion

#### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

##### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en optimización de hiperparámetros

In [213]:
PARAM$trainingstrategy$final_train <- c( 202107,
  202106, 202105, 202104, 202103, 202102, 202101,
  202012, 202011, 202010, 202009, 202008, 202007,
  202006, 202005
)

dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

# creo el dfinal_train en formato  LightGBM
dfinal_train <- lgb.Dataset(
  data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
  label= dataset[fold_final_train == TRUE, clase01],
  free_raw_data= TRUE
)

nrow( dfinal_train) # verifico el tamaño

[1] 192651

##### Final Training Hyperparameters

In [214]:
# uno los parametros fijos y los mejores encontrados de los variables
fijos <- copy(PARAM$lgbm$param_fijos)

# quito lo que optimice en la Bayesian Optimization
fijos$num_iterations <- NULL
fijos$early_stopping_rounds <- NULL

# agrego a los hiperparametros fijos los que encontre con la Bayesian Optimization
param_final <- c(fijos, PARAM$out$lgbm$mejores_hiperparametros)

##### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

In [215]:
final_model <- lgb.train(
  data= dfinal_train,
  param= param_final,
  verbose= -100
)

In [216]:
# grabo a disco el modelo en un formato para seres humanos ... ponele ...

lgb.save(final_model, "modelo.txt")

In [217]:
# ahora imprimo la importancia de variables

tb_importancia <- as.data.table(lgb.importance(final_model))
archivo_importancia <- "impo.txt"

fwrite( tb_importancia,
  file= archivo_importancia,
  sep= "\t"
)

#### Scoring

Aplico el modelo final a los datos del futuro

In [218]:
PARAM$trainingstrategy$future <- c(202109)

dfuture <- dataset[ foto_mes %in% PARAM$trainingstrategy$future ]

In [219]:
# aplico final_model   a dfuture

prediccion <- predict(
  final_model,
  data.matrix(dfuture[, campos_buenos, with= FALSE])
)

##### Tabla Prediccion

In [220]:
tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob := prediccion]

# grabo las probabilidad del modelo
#  me va a ser util para hacer Ensembles de modelos
fwrite(tb_prediccion,
  file= "prediccion.txt",
  sep= "\t"
)

#### Kaggle Competition Submit

Genero las salidas y hago los submits a Kaggle
<br>El notebook esta preparado para la Modalidad Gerencial, los analistas deben hacer cambios.


In [221]:
# genero archivos con los  "envios" mejores
# suba TODOS los archivos a Kaggle

PARAM$kaggle$competencia <- "utn-2026-virtual-mgr"
PARAM$kaggle$cortes <- seq(800, 1300, by = 50)

# ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle")

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marclo los primeros

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_", envios, ".csv")

  # grabo el archivo
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  # subida a Kaggle, armo la linea de comando
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste( "-f", archivo_kaggle)

  mensaje <- paste0("-m 'envios=", envios,
  "  semilla=", PARAM$semilla_primigenia,
    "'" )

  linea <- paste( comando, competencia, arch, mensaje)

  Sys.sleep(30)
  salida <- system(linea, intern=TRUE) # el submit a Kaggle
  cat(salida, "\n")
}

In [222]:
# grabo los parametros
if( !require("yaml")) install.packages("yaml")
require("yaml")

write_yaml( PARAM, file="PARAM.yml")

In [223]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Mon Sep 21 12:04:29 AM 2026"